# HW2: reproducing super-convergence

ResNet-56 on CIFAR-10 from Smith & Topin, [Super-Convergence: Very Fast Training of Neural Networks Using Large Learning Rates](https://arxiv.org/abs/1708.07120), with the settings from the authors' Caffe files ([lnsmith54/super-convergence](https://github.com/lnsmith54/super-convergence)): FP32, batch of 1000 images split as the paper's 8 GPUs x 125, full run lengths. Code and details: [hw2/](https://github.com/PE51K/itmo-ai-talent-hub-efficient-models-course/tree/main/hw2). Runtime: Kaggle, GPU T4 x2.

Terms from the paper used below:

- **Iteration**: one SGD update on a batch of 1000 images. Run lengths and LR schedules are counted in iterations; one pass over 50,000 training images is 50 iterations.
- **LR range test** (Section 3): a run where LR starts at 0 and grows linearly. Test accuracy vs LR shows which learning rates the network can still train with. The paper uses it to check whether super-convergence is possible.
- **CLR, cyclical learning rate** (Section 3): LR goes linearly from a minimum to a maximum bound over `stepsize` iterations, then back over the next `stepsize` iterations. The two steps make one cycle. Caffe calls this policy `triangular`.
- **Super-convergence** (Section 1): one CLR cycle up to a very large LR (here 3, typical LRs are about 0.1) reaches higher test accuracy in far fewer iterations than typical training.
- **PC-LR, piecewise constant learning rate** (Section 2, the "typical, standard, or a piecewise-constant training regime"): the baseline. LR 0.35, divided by 10 at iterations 50,000 and 70,000, 80,000 iterations in total. Caffe calls this policy `multistep`.
- **BN MAF**: Caffe's `moving_average_fraction` of batch norm (Table 1). Test-time BN uses running mean and variance, and MAF is how much of the old value is kept at each iteration. The paper uses 0.999 for PC-LR and 0.95 for the short CLR runs, because 0.999 follows the weights too slowly when training is fast.

| # | Run | Paper | LR policy | Training samples | Iterations | BN MAF | Paper result | T4 time, estimate |
|---|---|---|---|---|---|---|---|---|
| 1 | `lr_range_test` | Fig. 2b, "Max Iter=5k" | LR 0 -> 3 linearly | 50,000 | 5,000 | 0.95 | accuracy vs LR curve | ~40 min |
| 2 | `clr` | Fig. 1a | CLR 0.1-3, stepsize 5,000 | 50,000 | 10,000 | 0.95 | 92.4 % | ~1.5 h |
| 3 | `clr_10k` | Table 1 | CLR 0.1-3, stepsize 5,000 | 10,000 | 10,000 | 0.95 | 80.6 % | ~1.5 h |
| 4 | `pc_lr` | Fig. 1a | PC-LR 0.35 | 50,000 | 80,000 | 0.999 | 91.2 % | ~11 h |
| 5 | `pc_lr_10k` | Table 1 | PC-LR 0.35 | 10,000 | 80,000 | 0.999 | 71.4 % | ~11 h |

Times are not measured: FP16 runs of this code took 0.21-0.24 s per iteration on a T4, FP32 should be about 2x slower. Every run prints elapsed time and ETA every 1,000 iterations.

## Kaggle settings

Before running, in the notebook's Settings panel:

- Accelerator: GPU T4 x2. Each GPU trains its own run, so two runs go at once.
- Internet: on (Kaggle asks for phone verification first). The notebook clones the repo, installs Lightning, downloads CIFAR-10 and the authors' logs.

A Kaggle session is stopped after 12 hours, and the five runs need about 25 T4-hours, so they are split into two versions. Set `VERSION` in the next cell and click Save Version -> Save & Run All. A saved version runs in the background, the browser can be closed. Start with version 1, the long one, then set `VERSION = 2` and save another version.

| `VERSION` | GPU 0 | GPU 1 | Time, estimate |
|---|---|---|---|
| 1 | run 4 `pc_lr` | run 5 `pc_lr_10k` | ~11 h |
| 2 | run 1 `lr_range_test`, then run 2 `clr` | run 3 `clr_10k` | ~2.5 h |

`MAX_HOURS` stops a run after 11.5 hours of training even if it hasn't reached its last iteration, so a version always ends inside the 12-hour limit and keeps its output. A run appends a row to its CSV after every test (every 100 iterations), so a stopped run keeps its curve up to that point.

Results are written to `/kaggle/working/results`, the version's output (Output tab of the saved version), and the last cell packs them into `results.zip`. Each version plots only its own runs. To plot all five, unzip both `results.zip` into `hw2/` locally and run `uv run python plots.py`.

In [ ]:
VERSION = 1 # 1: PC-LR runs, 2: LR range test and CLR runs, see the table above
QUEUES = { # runs per GPU, each GPU trains its runs one after another
    1: ("pc_lr", "pc_lr_10k"),
    2: ("lr_range_test clr", "clr_10k"),
}
MAX_HOURS = 11.5 # per run, leaves time for setup and plots inside the 12-hour session

In [ ]:
!git clone --depth 1 https://github.com/PE51K/itmo-ai-talent-hub-efficient-models-course.git /tmp/repo || git -C /tmp/repo pull
%cd /tmp/repo/hw2
!pip install -q lightning
# /kaggle/working is the version's output, the repo and CIFAR-10 stay outside it and only results/ is linked into it
!mkdir -p /kaggle/working/results && ln -sfn /kaggle/working/results results

In [ ]:
import torch

assert torch.cuda.device_count() == 2, "set Accelerator to GPU T4 x2 in Settings"

In [ ]:
# Both runs of a version would otherwise download CIFAR-10 into the same folder at the same time
!python -c "from train import load_cifar; load_cifar(train=True); load_cifar(train=False)"

## Runs

### 1. LR range test (Fig. 2b)

Section 3: "The LR range test can be used to determine if super-convergence is possible for an architecture." LR grows linearly from 0 to 3 over 5,000 iterations (`triangular` policy with `base_lr` 0, `max_lr` 3, `stepsize` = `max_iter` = 5,000). In the paper's Fig. 2b test accuracy stays at 70-80 % up to LR 3, an order of magnitude above usual LRs, and the authors read this as a sign that super-convergence is possible. Fig. 2b also has 20k and 100k iteration versions of the test, we run the 5k one.

### 2. Super-convergence on CIFAR-10 (Fig. 1a, CLR curve)

One CLR cycle: LR 0.1 -> 3 over 5,000 iterations, then 3 -> 0.1 over the next 5,000. 10,000 iterations are 200 epochs. BN MAF 0.95. Paper: 92.4 % test accuracy after 10,000 iterations, vs 91.2 % for PC-LR after 80,000 (run 4).

### 3. Super-convergence with 10,000 training samples (Table 1)

The same CLR run on 10,000 training images: the first 10,000 of CIFAR-10, since the paper doesn't say which ones. Paper: 80.6 % vs 71.4 % for PC-LR (run 5). Table 1 shows that the gap between CLR and PC-LR grows as the training set shrinks: 1.2 % at 50,000 samples, 9.2 % at 10,000.

### 4. Typical training on CIFAR-10 (Fig. 1a, LR=0.35 curve)

The PC-LR baseline: LR 0.35, x0.1 at iterations 50,000 and 70,000, 80,000 iterations (1,600 epochs), BN MAF 0.999. Paper: 91.2 %. The authors' own log of this run in the official repo ends at 90.3 %.

### 5. Typical training with 10,000 training samples (Table 1)

The same PC-LR run on the 10,000 images of run 3. Paper: 71.4 %. Section 5: "with a piecewise constant learning rate schedule the training encounters difficulties and diverges along the way". The authors' log of this run has test accuracy at 10-50 % until the LR drop at 50,000.

## Training

`CUDA_VISIBLE_DEVICES` gives each process one GPU. Progress lines of both processes are mixed in the output below, each starts with its run name.

In [ ]:
gpu0, gpu1 = QUEUES[VERSION]
# The first process goes to the background, wait keeps the cell running until both finish
!CUDA_VISIBLE_DEVICES=0 python train.py {gpu0} --max-hours {MAX_HOURS} & CUDA_VISIBLE_DEVICES=1 python train.py {gpu1} --max-hours {MAX_HOURS}; wait

## Plots

`plots.py` plots the runs found in `results/` next to the authors' Caffe logs of the same runs (from `Results/` of the official repo, downloaded on first use). It also prints final test accuracy from the paper, from the authors' log and from our run, and how many iterations each run reached.

In [ ]:
!python plots.py

In [ ]:
from pathlib import Path

from IPython.display import Image, display

for path in sorted(Path("results/figures").glob("*.png")):
    display(Image(str(path)))

In [ ]:
import shutil

shutil.make_archive("/kaggle/working/results", "zip", root_dir="/kaggle/working", base_dir="results")